# TEST-01: Buy-and-Hold Verification

**WP-5a Pipeline Audit** | 2026-04-16

**Intent:** Verify that `run_backtest` on a constant +1 signal reproduces a plausible buy-and-hold result for BTCUSDT 4h over the full evaluation window (2020-01-01 to 2026-04-11).

**Expected (derived independently):**
- Final equity ≈ `initial_equity - entry_costs + (last_close - entry_open)`
- Trade count = 0 (constant +1 never triggers a close; no completed round-trips)
- Max drawdown matches peak-to-trough of independently computed equity curve
- No liquidation events

In [1]:
import sys, math
from pathlib import Path
import pandas as pd
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')
DATA_ROOT = Path('C:/GitHub/crypto-perp-wedge/data/processed')
GAPS_PATH = Path('C:/GitHub/crypto-perp-wedge/data/irreparable_gaps.json')

from lib.backtest import run_backtest
from lib.evaluation import max_drawdown

OHLCV_PATH = DATA_ROOT / 'binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet'
COSTS = {'fee_bps_taker': 4.0, 'fee_bps_maker': 2.0, 'slippage_bps': 2.0, 'initial_equity': 10_000.0}

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df['close_time_ms'].astype('int64'), unit='ms', utc=True)
    return df[['open','high','low','close','volume']].astype(float).sort_index()

ohlcv = load_ohlcv(OHLCV_PATH)
funding = pd.DataFrame({'funding_rate': []}, index=pd.DatetimeIndex([], tz='UTC'))
sig = pd.Series(1.0, index=ohlcv.index, name='signal')
result = run_backtest(sig, ohlcv, funding, **COSTS, execution='next_bar_open',
                      stream='klines', interval='4h', gaps_path=GAPS_PATH)
print(f'4h bars: {len(ohlcv)}')
print(f'Range: {ohlcv.index[0]} to {ohlcv.index[-1]}')
print(f'Entry open (bar 1): {ohlcv["open"].iloc[1]:.2f}')
print(f'Last close: {ohlcv["close"].iloc[-1]:.2f}')

4h bars: 13758
Range: 2020-01-01 03:59:59.999000+00:00 to 2026-04-11 23:59:59.999000+00:00
Entry open (bar 1): 7221.80
Last close: 73013.40


In [2]:
# Independent calculation
entry_price = ohlcv['open'].iloc[1]  # fill at open of bar 1 (signal at bar 0 = +1)
last_close = ohlcv['close'].iloc[-1]
fee_entry = entry_price * 4.0 / 10_000.0
slip_entry = entry_price * 2.0 / 10_000.0
expected_final = (10_000.0 - fee_entry - slip_entry) + (last_close - entry_price)

# Independent max-drawdown: track equity curve from bar 1
eq_ind = [(10_000.0 - fee_entry - slip_entry) + (ohlcv['close'].iloc[i] - entry_price)
          for i in range(1, len(ohlcv))]
eq_s = pd.Series(eq_ind)
peak_s = eq_s.cummax()
dd_s = (eq_s - peak_s) / peak_s
ind_max_dd = float(-dd_s.min())

actual_final = float(result.equity_curve.iloc[-1])
actual_dd, _, _ = max_drawdown(result.equity_curve)

print(f'=== Equity comparison ===')
print(f'Expected: {expected_final:.5f}')
print(f'Actual:   {actual_final:.5f}')
print(f'Diff:     {abs(expected_final - actual_final):.6f} (pass if < $1.00)')
print()
print(f'=== Max drawdown comparison ===')
print(f'Independent: {ind_max_dd*100:.4f}%')
print(f'Backtester:  {actual_dd*100:.4f}%')
print(f'Diff:        {abs(ind_max_dd - actual_dd)*100:.4f}pp (pass if < 0.5pp)')
print()
print(f'=== Trade counting ===')
print(f'trades_df rows: {len(result.trades_df)} (expected: 0 -- no closed round-trip)')
print(f'fees_paid:      {result.fees_paid:.5f} (entry side only, no exit fee)')
print(f'unrealized_pnl: {result.unrealized_pnl:.2f}')
print(f'skipped_bars:   {result.skipped_bars}')

=== Equity comparison ===
Expected: 75787.26692
Actual:   75787.26692
Diff:     0.000000 (pass if < $1.00)

=== Max drawdown comparison ===
Independent: 74.0858%
Backtester:  74.0858%
Diff:        0.0000pp (pass if < 0.5pp)

=== Trade counting ===
trades_df rows: 0 (expected: 0 -- no closed round-trip)
fees_paid:      2.88872 (entry side only, no exit fee)
unrealized_pnl: 65791.60
skipped_bars:   0


In [3]:
equity_ok = abs(expected_final - actual_final) < 1.0
dd_ok = abs(ind_max_dd - actual_dd) < 0.005
trade_ok = len(result.trades_df) == 0
verdict = 'PASS' if (equity_ok and dd_ok and trade_ok) else 'FAIL'

print(f'=== VERDICT: {verdict} ===')
print(f'Equity match:    {"PASS" if equity_ok else "FAIL"}')
print(f'Max DD match:    {"PASS" if dd_ok else "FAIL"}')
print(f'Trade count = 0: {"PASS" if trade_ok else "FAIL"}')
print()
print('Semantic note: trades_df only records COMPLETED round-trips.')
print('An unclosed long at end-of-run is tracked via unrealized_pnl and equity_curve')
print('but is not in trades_df. This is the repo convention (not a bug).')

=== VERDICT: PASS ===
Equity match:    PASS
Max DD match:    PASS
Trade count = 0: PASS

Semantic note: trades_df only records COMPLETED round-trips.
An unclosed long at end-of-run is tracked via unrealized_pnl and equity_curve
but is not in trades_df. This is the repo convention (not a bug).
